# Assignment 1 — Q1–Q3 可執行版本

本輪 **RUN_STAGE = "q3_remaining"**：沿用已驗證的 Version 11 Rank16/default 結果，只從固定 revision 的原始 Llama-3.2-1B 分別訓練 quarter、half、bottom。三組正式評估完成後，與 v11/default 合併產生四組 Q3 圖表；CUDA allocator 使用 native 預設以對齊 v11。新增三組的來源版本在保存 Kaggle notebook 時指定。

- 使用課程設定：25,000 筆原始 MetaMathQA，截斷後保留有 response token 的列；每個評估資料集 1,000 題；FP16、1 epoch、LR 2e-5、global batch 128、最多 256 個輸出 token。
- Q2 已加入純訓練計時；Q3 已補齊 quarter/half/bottom 奇異值區段。
- Kaggle Settings：GPU T4 x2、Internet On；Add-ons → Secrets：勾選 HF_TOKEN。
- 設定區可改 RUN_STAGE="q3_remaining"/"rank64"/"rank256"/"q1_q2"/"q3"/"all"。每次 Save & Run All 產生獨立版本，請下載並保留各版本 Output。
- 輸出在 /kaggle/working/Assignment1；benchmark_results.csv 的 train_minutes 為 Q2 訓練時間。只有 status="ok" 且兩個 total=1000 的列才能用於報告。

修改自[課程官方程式](https://github.com/nycubigdata-2026fall/BigData2026-Assignment1/tree/b1521186a8a516a4e81482d5e6ad5f587d9aae52)。尚未執行完成前沒有作業實驗結果。

Progress: explicit stdout JSON lines every 60 seconds, on phase transitions, after the first evaluation batch, and every 100 evaluated examples. Heartbeats include completed counts and seconds since the last completed batch; they do not imply forward progress. Task results are saved after each evaluation task.

Ponytail Ultra 精簡版：只更新程式；既有 v11–v15 實測與模型仍保留原來源。此來源以 Quick Save 保存，不代表新 GPU 實驗。


In [ ]:
# Fail early before installing anything if the execution session has no CUDA.
import json, os, subprocess, sys
from pathlib import Path
import threading
import time
from datetime import datetime, timezone

def save_json(path, value):
    path = Path(path)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return path

PROGRESS_INTERVAL_SECONDS = 60
PROGRESS_PATH = Path("/kaggle/working/Assignment1/progress.json")
PROGRESS_PATH.parent.mkdir(parents=True, exist_ok=True)
_progress_lock = threading.Lock()
_progress_stop = threading.Event()
_progress_state = {}
_progress_phase_started = time.perf_counter()
_progress_last_completion = _progress_phase_started

def _emit_progress_locked():
    now = time.perf_counter()
    snapshot = {
        **_progress_state,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "phase_elapsed_seconds": round(now - _progress_phase_started, 1),
        "seconds_since_progress": round(now - _progress_last_completion, 1),
    }
    if "/train/" in snapshot.get("stage", "") and "torch" in globals():
        try:
            snapshot["cuda_memory"] = cuda_memory_snapshot()
        except Exception as diagnostic_error:
            snapshot["cuda_memory_error_type"] = type(diagnostic_error).__name__
    total = snapshot.get("total")
    snapshot["phase_percent"] = (
        round(100 * snapshot["completed"] / total, 2) if total else None
    )
    temporary = PROGRESS_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(PROGRESS_PATH)
    print("PROGRESS " + json.dumps(snapshot, ensure_ascii=False), flush=True)

def report_progress(stage, completed=0, total=None, status="running", force=False, **details):
    global _progress_phase_started, _progress_last_completion
    with _progress_lock:
        now = time.perf_counter()
        changed_stage = stage != _progress_state.get("stage")
        if changed_stage:
            _progress_state.clear()
            _progress_phase_started = _progress_last_completion = now
        elif completed != _progress_state.get("completed"):
            _progress_last_completion = now
        changed_status = status != _progress_state.get("status")
        _progress_state.update(stage=stage, completed=completed, total=total, status=status, **details)
        if changed_stage or changed_status or force:
            _emit_progress_locked()

def _progress_heartbeat():
    while not _progress_stop.wait(PROGRESS_INTERVAL_SECONDS):
        with _progress_lock:
            if _progress_state.get("status") == "running":
                _emit_progress_locked()

# A heartbeat proves the reporter is alive; completed counts prove actual work advances.
report_progress("bootstrap")
threading.Thread(target=_progress_heartbeat, name="assignment1-progress", daemon=True).start()

# Match Version 11 Rank16: use native allocator defaults in a fresh session.
os.environ.pop('PYTORCH_ALLOC_CONF', None)
os.environ.pop('PYTORCH_CUDA_ALLOC_CONF', None)
print('CUDA allocator config: native defaults', flush=True)
import torch

def cuda_memory_snapshot():
    states = []
    for device_index in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
        states.append({
            'device': device_index,
            'allocated_mib': round(torch.cuda.memory_allocated(device_index) / 2**20, 1),
            'reserved_mib': round(torch.cuda.memory_reserved(device_index) / 2**20, 1),
            'peak_allocated_mib': round(torch.cuda.max_memory_allocated(device_index) / 2**20, 1),
            'free_mib': round(free_bytes / 2**20, 1),
            'total_mib': round(total_bytes / 2**20, 1),
        })
    return states

bootstrap_root = Path("/kaggle/working/Assignment1")
bootstrap_root.mkdir(parents=True, exist_ok=True)
bootstrap_report = {
    "cuda_allocator_config": os.environ.get("PYTORCH_ALLOC_CONF"),
    "torch_version": str(torch.__version__),
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
(bootstrap_root / "bootstrap_report.json").write_text(
    json.dumps(bootstrap_report, indent=2), encoding="utf-8"
)
print(json.dumps(bootstrap_report, indent=2), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError(
        "This execution session has no CUDA. In Kaggle Edit > Settings select GPU T4 x2 "
        "and start a fresh GPU session. Saved accelerator metadata alone is insufficient."
    ) from None

# Preserve Kaggle's CUDA PyTorch while installing a compatible Transformers 4.x stack.
constraint_path = Path("/kaggle/temp/assignment1_torch_constraint.txt")
constraint_path.parent.mkdir(parents=True, exist_ok=True)
constraint_path.write_text("torch==" + str(torch.__version__) + "\n", encoding="utf-8")
report_progress("install_dependencies")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    "--timeout", "20", "--retries", "1", "--constraint", str(constraint_path),
    "transformers==4.56.2", "datasets==4.1.1", "accelerate==1.10.1",
    "huggingface_hub==0.34.4", "safetensors", "numpy<3", "pandas<3",
    "matplotlib<4", "tqdm",
], check=True)

report_progress("dependencies_ready", status="completed")


In [ ]:
import os
# Keep Hugging Face cache/authentication outside saved Kaggle outputs.
os.environ["HF_HOME"] = "/kaggle/temp/assignment1_hf"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Python/PyTorch ready')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
RESULT_ROOT = Path('/kaggle/working/Assignment1')
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'meta-llama/Llama-3.2-1B'
MODEL_REVISION = '4e20de362430cd3b72f300e6b0f18e50e7166e08'  # Same original model as the completed baseline and Rank 16.
MODEL_DTYPE = torch.float16
TRAIN_DEVICE = torch.device('cuda:0')
SVD_DEVICE = torch.device('cuda:0')

METAMATH_TRAIN_LIMIT = 25000
METAMATH_TRAIN_DATASET_ID = 'meta-math/MetaMathQA'

PISSA_EVAL_DATASET_ID = 'fxmeng/pissa-dataset'
PISSA_EVAL_DATA_DIR = 'metamath'
EVAL_TASKS = ['gsm8k', 'math']

MAX_LENGTH = 512
EPOCHS = 1
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
TARGET_GLOBAL_BATCH = 128

MICRO_BATCH_SIZE = 8
USE_DATA_PARALLEL = torch.cuda.device_count() >= 2
GRAD_ACCUM_STEPS = max(1, TARGET_GLOBAL_BATCH // MICRO_BATCH_SIZE)

EVAL_MAX_NEW_TOKENS = 256
EVAL_BATCH_SIZE = 4

# Save individual prediction JSONL files. Disable to reduce disk usage.
SAVE_PREDICTIONS = True
SAVE_ADAPTERS = True
SAVE_FULL_MODEL = True

# First round: official baseline + rank 16/default.
# Later options: q3_remaining, remaining_ranks, rank64, rank256, q1_q2, q3, all.
RUN_STAGE = "q3_remaining"
CURRENT_NOTEBOOK_VERSION = 17  # Set when the Kaggle source version is assigned at save.
RUN_PRESETS = {
    "first_run": (True, [{"rank": 16, "component": "default"}]),
    "remaining_ranks": (False, [{"rank": r, "component": "default"} for r in (64, 256)]),
    "rank64": (False, [{"rank": 64, "component": "default"}]),
    "rank256": (False, [{"rank": 256, "component": "default"}]),
    "q1_q2": (True, [{"rank": r, "component": "default"} for r in (16, 64, 256)]),
    "q3_remaining": (False, [{"rank": 16, "component": c} for c in ("quarter", "half", "bottom")]),
    "q3": (False, [{"rank": 16, "component": c} for c in ("default", "quarter", "half", "bottom")]),
    "all": (True, [
        {"rank": r, "component": "default"} for r in (16, 64, 256)
    ] + [
        {"rank": 16, "component": c} for c in ("quarter", "half", "bottom")
    ]),
}
if RUN_STAGE not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_STAGE: {RUN_STAGE}")
RUN_FRESH_BASELINE, MANUAL_RUNS = RUN_PRESETS[RUN_STAGE]
EVAL_EXAMPLES_PER_TASK = 1000
# Training/evaluation sizes and hyperparameters remain the course settings.

assert TARGET_GLOBAL_BATCH % MICRO_BATCH_SIZE == 0
print('Effective batch:', MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS)
print('MetaMathQA train limit:', METAMATH_TRAIN_LIMIT)
print('Evaluation tasks:', EVAL_TASKS)
print('Results:', RESULT_ROOT)

print("Run stage:", RUN_STAGE)
print("RUN_FRESH_BASELINE:", RUN_FRESH_BASELINE)
print("MANUAL_RUNS:", MANUAL_RUNS)

ACTIVE_RUN_TAG = "setup"


In [ ]:
# Never print or save the token. Kaggle API authentication and HF model access are separate.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

PREFLIGHT_PATH = RESULT_ROOT / "preflight_report.json"
PREFLIGHT_REPORT = {
    "model_id": MODEL_ID, "torch_version": str(torch.__version__),
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "status": "checking",
}
def save_preflight_report():
    save_json(PREFLIGHT_PATH, PREFLIGHT_REPORT)

save_preflight_report()
if not torch.cuda.is_available():
    PREFLIGHT_REPORT.update(status="blocked", reason="GPU_NOT_ENABLED")
    save_preflight_report()
    raise RuntimeError("Select GPU T4 x2 in Kaggle Notebook Settings.") from None

report_progress("hf_access_check")
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if not hf_token:
        raise ValueError("empty")
except Exception as exc:
    PREFLIGHT_REPORT.update(status="blocked", reason="HF_SECRET_READ_FAILED",
                            error_type=type(exc).__name__)
    save_preflight_report()
    raise RuntimeError(
        "Cannot read the HF_TOKEN secret. Check Add-ons > Secrets attachment "
        "and Kaggle session connectivity to the secrets service."
    ) from None

os.environ["HF_TOKEN"] = hf_token
try:
    hf_model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION, token=hf_token)
    from huggingface_hub import hf_hub_download
    hf_hub_download(MODEL_ID, "config.json", revision=MODEL_REVISION, token=hf_token)
except Exception as exc:
    PREFLIGHT_REPORT.update(status="blocked", reason="HF_ACCESS_OR_INTERNET_FAILED",
                            error_type=type(exc).__name__)
    save_preflight_report()
    raise RuntimeError(
        "Cannot download gated Llama-3.2-1B config. Check Internet and the HF_TOKEN "
        "account's approved read access to meta-llama/Llama-3.2-1B."
    ) from None
PREFLIGHT_REPORT.update(hf_secret_available=True, model_access=True,
                        model_revision=hf_model_info.sha)
save_preflight_report()
del hf_token
print("HF_TOKEN is enabled and gated model access works.")

assert hf_model_info.sha == MODEL_REVISION, "Model revision differs from the completed baseline."
print("Pinned model revision:", MODEL_REVISION, flush=True)

report_progress("hf_access_ready", status="completed")


In [ ]:
report_progress("load_tokenizer")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.model_max_length = MAX_LENGTH

IGNORE_INDEX = -100
PISSA_PROMPT = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n\n### Response:'
)

print('Tokenizer:', MODEL_ID)
print('EOS:', tokenizer.eos_token_id, 'PAD:', tokenizer.pad_token_id)


In [ ]:
def load_metamathqa_train_like_pissa():
    split = f'train[:{METAMATH_TRAIN_LIMIT}]'
    train = load_dataset(
        METAMATH_TRAIN_DATASET_ID,
        split=split,
    )

    train = train.map(
        lambda x: {
            'instruction': str(x['query']).strip(),
            'output': str(x['response']).strip(),
            'source_type': str(x.get('type', '')),
        },
        remove_columns=train.column_names,
        desc='Formatting MetaMathQA for PiSSA training',
    )
    return train

def load_pissa_metamath_eval():
    raw_test = load_dataset(
        PISSA_EVAL_DATASET_ID,
        data_dir=PISSA_EVAL_DATA_DIR,
        split='test',
    )

    eval_sets = {}
    normalized_types = [str(x).lower() for x in raw_test['type']]

    for task in EVAL_TASKS:
        keep_indices = [
            i for i, task_type in enumerate(normalized_types)
            if task_type == task
        ]
        subset = raw_test.select(keep_indices)
        if len(subset) == 0:
            raise RuntimeError(
                f'No {task!r} rows found in '
                f'{PISSA_EVAL_DATASET_ID}/{PISSA_EVAL_DATA_DIR} test split.'
            )

        # PiSSA gen_vllm.py generates from "instruction" and passes "output"
        # unchanged to test_acc.py as the gold answer.
        subset = subset.map(
            lambda x: {
                'instruction': str(x['instruction']),
                'gold_answer': str(x['output']),
                'type': str(x['type']).lower(),
            },
            remove_columns=subset.column_names,
            desc=f'Formatting PiSSA {task} evaluation set',
        )
        eval_sets[task] = subset

    return eval_sets


In [ ]:
report_progress("load_datasets")
# Load once; all PiSSA ranks/components reuse exactly the same training and test rows.
TRAIN_DATASET = load_metamathqa_train_like_pissa()
EVAL_DATASETS = load_pissa_metamath_eval()

print('\nTRAIN:', TRAIN_DATASET)
print('train example:', TRAIN_DATASET[0])

for task, ds in EVAL_DATASETS.items():
    print(f'\nEVAL {task}:', ds)
    print('test example:', ds[0])

report_progress("datasets_ready", len(TRAIN_DATASET), len(TRAIN_DATASET), status="completed")


In [ ]:
report_progress("tokenize_training_data")
def tokenize_supervised_example(example):
    source = PISSA_PROMPT.format(instruction=example['instruction'])
    target = f"{example['output']}\n{tokenizer.eos_token}"

    full = tokenizer(
        source + target,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=True,
    )
    source_tok = tokenizer(
        source,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=True,
    )

    input_ids = full['input_ids']
    labels = input_ids.copy()
    source_len = min(len(source_tok['input_ids']), len(labels))
    labels[:source_len] = [IGNORE_INDEX] * source_len

    return {
        'input_ids': input_ids,
        'labels': labels,
        'supervised_tokens': sum(v != IGNORE_INDEX for v in labels),
    }

@dataclass
class ResponseOnlyCollator:
    pad_token_id: int

    def __call__(self, features):
        max_len = max(len(x['input_ids']) for x in features)
        input_ids, labels, attention_mask = [], [], []

        for x in features:
            n_pad = max_len - len(x['input_ids'])
            input_ids.append(x['input_ids'] + [self.pad_token_id] * n_pad)
            labels.append(x['labels'] + [IGNORE_INDEX] * n_pad)
            attention_mask.append([1] * len(x['input_ids']) + [0] * n_pad)

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        }

collator = ResponseOnlyCollator(tokenizer.pad_token_id)

TOKENIZED_TRAIN = TRAIN_DATASET.map(
    tokenize_supervised_example,
    remove_columns=TRAIN_DATASET.column_names,
    desc='Tokenizing MetaMathQA',
)
before = len(TOKENIZED_TRAIN)
TOKENIZED_TRAIN = TOKENIZED_TRAIN.filter(
    lambda x: x['supervised_tokens'] > 0,
    desc='Removing rows with no supervised tokens after truncation',
)
print('MetaMathQA train rows:', before, 'usable:', len(TOKENIZED_TRAIN))

report_progress("tokenized_data_ready", len(TOKENIZED_TRAIN), len(TOKENIZED_TRAIN), status="completed")


In [ ]:
def load_fresh_model(to_device=False):
    report_progress("load_model", run_tag=ACTIVE_RUN_TAG)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        dtype=MODEL_DTYPE,
        device_map='cpu',
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = True
    if to_device:
        model = model.to(TRAIN_DEVICE)
    report_progress("model_ready", status="completed", run_tag=ACTIVE_RUN_TAG)
    return model

def cleanup_model(model=None):
    if model is not None:
        try:
            model.to('cpu')
        except Exception:
            pass
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
class PiSSALinear(nn.Module):
    def __init__(self, weight_residual, pissa_A, pissa_B, bias=None):
        super().__init__()
        out_features, in_features = weight_residual.shape
        rank = pissa_A.shape[1]
        assert pissa_A.shape == (out_features, rank)
        assert pissa_B.shape == (rank, in_features)

        self.rank = rank
        self.weight = nn.Parameter(weight_residual.contiguous(), requires_grad=False)
        self.pissa_A = nn.Parameter(pissa_A.float())
        self.pissa_B = nn.Parameter(pissa_B.float())
        if bias is None:
            self.register_parameter('bias', None)
        else:
            self.bias = nn.Parameter(bias.detach().clone(), requires_grad=False)

    def forward(self, x):
        frozen_out = F.linear(x, self.weight, self.bias)
        with torch.autocast(device_type=x.device.type, enabled=False):
            hidden = F.linear(x.float(), self.pissa_B)
            update = F.linear(hidden, self.pissa_A)
        return frozen_out + update.to(frozen_out.dtype)


In [ ]:
@dataclass
class PiSSAInitStats:
    name: str
    rank: int
    component: str
    relative_error_stored: float

@torch.no_grad()
def pissa_from_linear_exact(linear, rank, component, layer_name=''):
    W = linear.weight.detach().to(SVD_DEVICE, dtype=torch.float32)
    out_features, in_features = W.shape
    k = min(out_features, in_features)
    if rank > k:
        raise ValueError(f'{layer_name}: rank={rank} > max rank={k}')

    U, S, Vh = torch.linalg.svd(W, full_matrices=False)

    # Q3: singular values returned by SVD are ordered from largest to smallest.
    if component == "default":
        start = 0
    elif component == "quarter":
        start = len(S) // 4
    elif component == "half":
        start = len(S) // 2
    elif component == "bottom":
        start = len(S) - rank
    else:
        raise ValueError(f"Unknown component: {component}")
    end = start + rank
    if rank <= 0 or start < 0 or end > len(S):
        raise ValueError(f"{layer_name}: invalid SVD slice [{start}:{end}]")

    U_r, S_r, Vh_r = U[:, start:end], S[start:end], Vh[start:end, :]
    sqrt_S = torch.sqrt(S_r)
    A = U_r * sqrt_S.unsqueeze(0)
    B = sqrt_S.unsqueeze(1) * Vh_r
    W_res = W - (A @ B)

    stored = W_res.to(linear.weight.dtype).cpu()
    A_cpu, B_cpu, W_cpu = A.cpu(), B.cpu(), W.cpu()
    rel_err = (
        torch.linalg.vector_norm(W_cpu - (stored.float() + A_cpu @ B_cpu))
        / torch.linalg.vector_norm(W_cpu)
    ).item()

    replacement = PiSSALinear(
        stored,
        A_cpu,
        B_cpu,
        None if linear.bias is None else linear.bias.detach().cpu(),
    )
    stats = PiSSAInitStats(layer_name, rank, component, rel_err)

    del W, U, S, Vh, U_r, S_r, Vh_r, sqrt_S, A, B, W_res
    torch.cuda.empty_cache()
    gc.collect()
    return replacement, stats


In [ ]:
TARGET_MODULE_NAMES = {
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
}

def get_parent_module(root, full_name):
    if '.' not in full_name:
        return root, full_name
    parent_name, child_name = full_name.rsplit('.', 1)
    return root.get_submodule(parent_name), child_name

def inject_pissa(model, rank, component):
    model.requires_grad_(False)
    targets = [(name, layer) for name, layer in model.named_modules()
               if isinstance(layer, nn.Linear) and name.rsplit('.', 1)[-1] in TARGET_MODULE_NAMES]
    stats = []

    phase = f"{ACTIVE_RUN_TAG}/svd_init"
    report_progress(phase, 0, len(targets), run_tag=ACTIVE_RUN_TAG)
    for idx, (name, linear) in enumerate(targets, 1):
        report_progress(phase, idx - 1, len(targets), run_tag=ACTIVE_RUN_TAG, current_module=name)
        replacement, st = pissa_from_linear_exact(
            linear, rank=rank, component=component, layer_name=name
        )

        parent, child = get_parent_module(model, name)
        setattr(parent, child, replacement)
        stats.append(st)
        report_progress(
            phase, idx, len(targets), run_tag=ACTIVE_RUN_TAG, current_module=name,
            force=idx == 1 or idx % 7 == 0 or idx == len(targets),
        )

    return stats

def verify_trainables(model):
    suffixes = ('pissa_A', 'pissa_B')
    trainable, total, unexpected = 0, 0, []
    for name, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
            if not name.endswith(suffixes):
                unexpected.append(name)
    assert trainable > 0
    assert not unexpected, 'Unexpected trainables: ' + ', '.join(unexpected)
    print('Trainable parameters:', f'{trainable:,}', f'({100*trainable/total:.3f}%)')
    return trainable, total

@torch.no_grad()
def merge_pissa_and_unload(model):
    targets = [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, PiSSALinear)
    ]
    if not targets:
        raise RuntimeError('No PiSSALinear modules found to merge.')

    merged_names = []
    for name, layer in targets:
        merged_weight = (
            layer.weight.float()
            + layer.pissa_A.float() @ layer.pissa_B.float()
        ).to(dtype=layer.weight.dtype)

        replacement = nn.Linear(
            in_features=layer.weight.shape[1],
            out_features=layer.weight.shape[0],
            bias=layer.bias is not None,
            device=layer.weight.device,
            dtype=layer.weight.dtype,
        )
        replacement.weight.copy_(merged_weight)
        replacement.weight.requires_grad_(False)

        if layer.bias is not None:
            replacement.bias.copy_(layer.bias.to(
                device=layer.weight.device,
                dtype=layer.weight.dtype,
            ))
            replacement.bias.requires_grad_(False)

        parent, child = get_parent_module(model, name)
        setattr(parent, child, replacement)
        merged_names.append(name)

    remaining = [
        name for name, module in model.named_modules()
        if isinstance(module, PiSSALinear)
    ]
    if remaining:
        raise RuntimeError(
            'PiSSA merge incomplete; remaining modules: ' + ', '.join(remaining)
        )

    print(f'Merged {len(merged_names)} PiSSA layers into standard nn.Linear layers.')
    return merged_names


In [ ]:
class LossOnlyWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, **batch):
        return self.base_model(**batch).loss

def build_optimizer_and_scheduler(model, num_batches):
    trainable = [p for p in model.parameters() if p.requires_grad]
    updates_per_epoch = math.ceil(num_batches / GRAD_ACCUM_STEPS)
    total_updates = EPOCHS * updates_per_epoch
    warmup_updates = max(1, math.ceil(total_updates * WARMUP_RATIO))

    optimizer = torch.optim.AdamW(
        trainable,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    def lr_lambda(step):
        if step < warmup_updates:
            return float(step + 1) / float(max(1, warmup_updates))
        progress = (step - warmup_updates) / max(1, total_updates - warmup_updates)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler, total_updates, warmup_updates

def train_one_task(model, train_dataset, task_name):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        train_dataset,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
    )

    optimizer, scheduler, total_updates, warmup_updates = build_optimizer_and_scheduler(
        model, len(loader)
    )

    print(
        f'Training {task_name}: {len(train_dataset)} examples, '
        f'{len(loader)} micro-batches, {total_updates} updates, '
        f'{warmup_updates} warmup updates.'
    )

    model.config.use_cache = False
    model.train()
    loss_model = LossOnlyWrapper(model)
    train_model = (
        nn.DataParallel(loss_model, device_ids=[0, 1], output_device=0)
        if USE_DATA_PARALLEL
        else loss_model
    )

    scaler = torch.amp.GradScaler('cuda', enabled=True)
    optimizer.zero_grad(set_to_none=True)
    history = []
    update_step = 0
    running_loss = 0.0

    # Q2: time only the training loop, excluding SVD, model load, merge and evaluation.
    for device_index in range(2 if USE_DATA_PARALLEL else 1):
        torch.cuda.synchronize(device_index)
    phase = f"{ACTIVE_RUN_TAG}/train/{task_name}"
    report_progress(phase, 0, EPOCHS * len(loader), run_tag=ACTIVE_RUN_TAG, optimizer_updates=0, total_optimizer_updates=total_updates)
    train_t0 = time.perf_counter()

    completed_batches = 0
    micro_step = 0
    epoch = 0
    failed_operation = 'setup'
    try:
        for epoch in range(EPOCHS):
            for micro_step, batch in enumerate(loader, 1):
                failed_operation = 'batch_transfer'
                batch = {k: v.to(TRAIN_DEVICE, non_blocking=True) for k, v in batch.items()}

                failed_operation = 'forward'
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    loss = train_model(**batch)
                    if loss.ndim > 0:
                        loss = loss.mean()
                    scaled_loss = loss / GRAD_ACCUM_STEPS

                failed_operation = 'backward'
                scaler.scale(scaled_loss).backward()
                running_loss += float(loss.detach().cpu())

                should_step = (
                    micro_step % GRAD_ACCUM_STEPS == 0
                    or micro_step == len(loader)
                )
                if should_step:
                    failed_operation = 'optimizer_step'
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()
                    update_step += 1

                    mean_loss = running_loss / (
                        GRAD_ACCUM_STEPS
                        if micro_step % GRAD_ACCUM_STEPS == 0
                        else (micro_step - (update_step - 1) * GRAD_ACCUM_STEPS)
                    )
                    history.append({
                        'update': update_step,
                        'loss': mean_loss,
                        'lr': scheduler.get_last_lr()[0],
                    })
                    running_loss = 0.0
                completed_batches = epoch * len(loader) + micro_step
                report_progress(
                    phase, completed_batches, EPOCHS * len(loader),
                    run_tag=ACTIVE_RUN_TAG, optimizer_updates=update_step,
                    total_optimizer_updates=total_updates,
                    latest_loss=history[-1]["loss"] if history else None,
                    force=completed_batches == 1 or (should_step and update_step == 1),
                    sequence_length=int(batch["input_ids"].shape[1]),
                )

    except torch.OutOfMemoryError:
        try:
            diagnostic = {
                'run_tag': ACTIVE_RUN_TAG,
                'completed_microbatches': completed_batches,
                'attempted_microbatch': epoch * len(loader) + micro_step,
                'optimizer_updates': update_step,
                'failed_operation': failed_operation,
                'sequence_length': int(batch['input_ids'].shape[1]) if 'batch' in locals() and 'input_ids' in batch else None,
                'cuda_allocator_config': os.environ.get('PYTORCH_ALLOC_CONF'),
                'cuda_memory': cuda_memory_snapshot(),
            }
            save_json(RESULT_ROOT / f'{ACTIVE_RUN_TAG}_oom.json', diagnostic)
            report_progress(phase, completed_batches, EPOCHS * len(loader),
                            status='error', error_type='OutOfMemoryError', **diagnostic)
        except Exception as diagnostic_error:
            print('OOM diagnostic failed:', type(diagnostic_error).__name__, flush=True)
        raise

    for device_index in range(2 if USE_DATA_PARALLEL else 1):
        torch.cuda.synchronize(device_index)
    train_seconds = time.perf_counter() - train_t0
    report_progress(phase, EPOCHS * len(loader), EPOCHS * len(loader), status="completed", run_tag=ACTIVE_RUN_TAG, optimizer_updates=update_step, total_optimizer_updates=total_updates, train_seconds=train_seconds)
    print(f"Training time: {train_seconds / 60:.2f} minutes", flush=True)
    model.config.use_cache = True
    return history, train_seconds


In [ ]:
def remove_right_units(string):
    if "\\text{ " not in string:
        return string
    parts = string.split("\\text{ ")
    assert len(parts) == 2
    return parts[0]

def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    parts = string.split("\\sqrt")
    result = parts[0]
    for part in parts[1:]:
        result += "\\sqrt" + (part if part[0] == "{" else "{" + part[0] + "}" + part[1:])
    return result

def fix_fracs(string):
    parts = string.split("\\frac")
    result = parts[0]
    for part in parts[1:]:
        result += "\\frac"
        if part[0] == "{":
            result += part
            continue
        if len(part) < 2:
            return string
        a, b = part[:2]
        result += "{" + a + "}" + (b if b == "{" else "{" + b + "}") + part[2:]
    return result

def fix_a_slash_b(string):
    parts = string.split("/")
    if len(parts) != 2:
        return string
    try:
        a, b = map(int, parts)
        return f"\\frac{{{a}}}{{{b}}}" if string == f"{a}/{b}" else string
    except ValueError:
        return string

def strip_pissa_math_string(string):
    string = str(string)
    for old, new in (("\n", ""), ("\\!", ""), ("\\\\", "\\"), ("tfrac", "frac"),
                     ("dfrac", "frac"), ("\\left", ""), ("\\right", ""),
                     ("^{\\circ}", ""), ("^\\circ", ""), ("\\$", "")):
        string = string.replace(old, new)
    string = remove_right_units(string).replace("\\%", "").replace(" .", " 0.").replace("{.", "{0.")
    if not string:
        return string
    if string.startswith("."):
        string = "0" + string
    parts = string.split("=")
    if len(parts) == 2 and len(parts[0]) <= 2:
        string = parts[1]
    string = fix_fracs(fix_sqrt(string).replace(" ", ""))
    return fix_a_slash_b("\\frac{1}{2}" if string == "0.5" else string)

def pissa_math_is_equiv(str1, str2):
    if str1 is None or str2 is None:
        return str1 is str2
    try:
        return strip_pissa_math_string(str1) == strip_pissa_math_string(str2)
    except Exception:
        return str1 == str2

def normalize_gsm8k_gold(value):
    text = str(value).strip().replace(",", "")
    if "####" in text:
        text = text.split("####")[-1].strip()
    match = re.search(r"[\-+]?\d+(?:\.\d+)?", text)
    if not match:
        raise ValueError(f"Cannot parse GSM8K gold: {value!r}")
    return float(match.group())

def extract_last_boxed(text):
    text = str(text)
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    start += len("\\boxed{")
    depth = 1
    for index in range(start, len(text)):
        depth += (text[index] == "{") - (text[index] == "}")
        if depth == 0:
            return text[start:index].strip()
    return None

def clean_robust_answer_text(text):
    text = str(text).strip().strip("$")
    for wrapper in ("\\(", "\\)", "\\[", "\\]"):
        text = text.replace(wrapper, "")
    text = text.strip()
    while text.endswith((".", ",", ";")):
        text = text[:-1].strip()
    return text

def extract_robust_answer_text(completion):
    text = str(completion)
    matches = list(re.finditer(r"(?:the\s+)?(?:final\s+)?answer\s*(?:is\s*)?(?::|=)?\s*([^\n]+)",
                              text, flags=re.IGNORECASE))
    if matches:
        candidate = matches[-1].group(1).strip()
        boxed = extract_last_boxed(candidate)
        return (clean_robust_answer_text(boxed), "answer_marker_boxed") if boxed is not None else (
            clean_robust_answer_text(candidate), "answer_marker")
    if "####" in text:
        candidate = text.rsplit("####", 1)[-1].strip()
        if candidate:
            candidate = candidate.splitlines()[0].strip()
            if candidate:
                return clean_robust_answer_text(candidate), "hash_marker"
    boxed = extract_last_boxed(text)
    return (clean_robust_answer_text(boxed), "boxed") if boxed is not None else (None, None)

def parse_numeric_token(token):
    if token is None:
        return None
    token = str(token).strip().replace(",", "")
    try:
        return float(Fraction(token)) if "/" in token else float(token)
    except Exception:
        return None

def extract_robust_gsm8k_number(completion):
    candidate, source = extract_robust_answer_text(completion)
    if candidate is None:
        candidate, source = str(completion), "full_text_fallback"
    pattern = r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+|\.\d+)(?:\.\d+)?(?:/\d+(?:\.\d+)?)?"
    matches = re.findall(pattern, candidate)
    if not matches:
        matches, source = re.findall(pattern, str(completion)), "last_number_fallback"
    return (parse_numeric_token(matches[-1]) if matches else None), source

def simple_math_numeric_value(text):
    if text is None:
        return None
    text = clean_robust_answer_text(text).replace(",", "")
    number = r"([+-]?(?:\d+(?:\.\d+)?|\.\d+))"
    for pattern in (r"\\frac\{" + number + r"\}\{" + number + r"\}", number + "/" + number):
        match = re.fullmatch(pattern, text)
        if match:
            numerator, denominator = map(float, match.groups())
            return numerator / denominator if denominator else None
    try:
        return float(text)
    except Exception:
        return None

def robust_math_is_equiv(pred, gold):
    if pred is None or gold is None:
        return False
    pred, gold = clean_robust_answer_text(pred), clean_robust_answer_text(gold)
    if pissa_math_is_equiv(pred, gold):
        return True
    pred_num, gold_num = simple_math_numeric_value(pred), simple_math_numeric_value(gold)
    return pred_num is not None and gold_num is not None and math.isclose(pred_num, gold_num, rel_tol=0, abs_tol=1e-9)

@torch.no_grad()
def evaluate_generation(model, dataset, task_name, max_examples=None):
    if task_name not in EVAL_TASKS:
        raise ValueError(f"Unsupported evaluation task: {task_name}")

    model.eval()
    model.config.use_cache = True

    n = len(dataset) if max_examples is None else min(len(dataset), max_examples)

    old_padding_side = tokenizer.padding_side
    old_truncation_side = tokenizer.truncation_side
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    correct = 0
    unparsed = 0
    records = []

    torch.cuda.synchronize()
    phase = f"{ACTIVE_RUN_TAG}/eval/{task_name}"
    report_progress(phase, 0, n, run_tag=ACTIVE_RUN_TAG, task=task_name)
    t0 = time.perf_counter()

    try:
        for start in range(0, n, EVAL_BATCH_SIZE):
            end = min(start + EVAL_BATCH_SIZE, n)
            report_progress(phase, start, n, run_tag=ACTIVE_RUN_TAG, task=task_name, active_batch_start=start, active_batch_end=end)
            rows = dataset[start:end]
            prompts = [str(x) for x in rows["instruction"]]

            enc = tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            enc = {k: v.to(TRAIN_DEVICE) for k, v in enc.items()}

            generated = model.generate(
                **enc,
                do_sample=False,
                max_new_tokens=EVAL_MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            prompt_width = enc["input_ids"].shape[1]
            decoded = tokenizer.batch_decode(
                generated[:, prompt_width:],
                skip_special_tokens=True,
            )

            for i, text in enumerate(decoded):
                gold_raw = rows["gold_answer"][i]

                if task_name == "gsm8k":
                    gold = normalize_gsm8k_gold(gold_raw)
                    pred, extraction_source = extract_robust_gsm8k_number(text)

                    ok = (
                        pred is not None
                        and math.isclose(
                            float(pred),
                            float(gold),
                            rel_tol=0.0,
                            abs_tol=1e-9,
                        )
                    )
                    parsed_answer = None if pred is None else str(pred)
                    normalized_gold = str(gold)
                    parsed = pred is not None

                else:
                    pred, extraction_source = extract_robust_answer_text(text)
                    ok = (
                        pred is not None
                        and robust_math_is_equiv(pred, gold_raw)
                    )
                    parsed_answer = None if pred is None else str(pred)
                    normalized_gold = str(gold_raw)
                    parsed = pred is not None

                if not parsed:
                    unparsed += 1

                correct += int(ok)
                records.append({
                    "task": task_name,
                    "instruction": prompts[i],
                    "prediction_text": text,
                    "pred_answer": parsed_answer,
                    "gold_answer": normalized_gold,
                    "correct": bool(ok),
                    "extraction_source": extraction_source,
                })
            report_progress(
                phase, end, n, run_tag=ACTIVE_RUN_TAG, task=task_name,
                partial_correct=correct, partial_accuracy=correct / end,
                active_batch_start=None, active_batch_end=None,
                force=start == 0 or end % 100 == 0 or end == n,
            )
    finally:
        tokenizer.padding_side = old_padding_side
        tokenizer.truncation_side = old_truncation_side

    torch.cuda.synchronize()
    eval_seconds = time.perf_counter() - t0
    report_progress(phase, n, n, status="completed", run_tag=ACTIVE_RUN_TAG, task=task_name, correct=correct, accuracy=correct / n if n else 0.0, eval_seconds=eval_seconds)

    return {
        "task": task_name,
        "accuracy": correct / n if n else 0.0,
        "correct": correct,
        "total": n,
        "unparsed": unparsed,
        "eval_seconds": eval_seconds,
        "records": records,
        "evaluation_protocol": (
            "robust_generation_numeric_match"
            if task_name == "gsm8k"
            else "robust_generation_math_equivalence"
        ),
    }


In [ ]:
def evaluate_all_tasks(model, max_examples=None):
    results = {}
    for task in EVAL_TASKS:
        results[task] = evaluate_generation(
            model,
            EVAL_DATASETS[task],
            task,
            max_examples=max_examples,
        )
        task_result = {k: v for k, v in results[task].items() if k != "records"}
        save_json(RESULT_ROOT / f'{ACTIVE_RUN_TAG}_{task}_evaluation.json', task_result)
        if SAVE_PREDICTIONS:
            write_jsonl(RESULT_ROOT / f"{ACTIVE_RUN_TAG}_{task}_predictions.jsonl", results[task]["records"])
    return results

def flatten_eval_results(results):
    return {f'{task}_{key}': row[key] for task, row in results.items()
            for key in ('accuracy', 'correct', 'total', 'unparsed', 'eval_seconds')}


In [ ]:
def write_jsonl(path, records):
    with Path(path).open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')

def safe_run_tag(method, rank=None, component=None):
    if method == 'fresh':
        return 'metamath_fresh'
    if method != 'pissa':
        raise ValueError(f'Unsupported method: {method}')
    return f'metamath_pissa_{component}_r{rank}'

def save_summary(summary, records=None):
    tag = summary['run_tag']
    path = RESULT_ROOT / f'{tag}_summary.json'
    save_json(path, summary)
    if SAVE_PREDICTIONS and records is not None:
        write_jsonl(RESULT_ROOT / f'{tag}_predictions.jsonl', records)
    return path

def save_master_results(rows):
    df = pd.DataFrame(rows)
    df.to_csv(RESULT_ROOT / 'benchmark_results.csv', index=False)
    save_json(RESULT_ROOT / 'benchmark_results.json', rows)
    return df


In [ ]:
def run_experiment(rank=None, component=None, max_eval_examples=1000):
    global ACTIVE_RUN_TAG
    method = 'fresh' if rank is None else 'pissa'
    tag = ACTIVE_RUN_TAG = safe_run_tag(method, rank, component)
    print('\n' + tag, flush=True)
    model = load_fresh_model(to_device=rank is None)
    summary = {
        'run_tag': tag, 'method': method, 'rank': rank, 'component': component,
        'training_dataset': METAMATH_TRAIN_DATASET_ID, 'seed': SEED,
        'training_examples': 0, 'init_seconds': 0.0, 'trainable_parameters': 0,
    }
    try:
        if rank is not None:
            torch.manual_seed(SEED)
            torch.cuda.manual_seed_all(SEED)
            torch.cuda.synchronize()
            start = time.perf_counter()
            init_stats = inject_pissa(model, rank=rank, component=component)
            torch.cuda.synchronize()
            init_seconds = time.perf_counter() - start
            trainable, total_params = verify_trainables(model)
            model = model.to(TRAIN_DEVICE)
            history, train_seconds = train_one_task(model, TOKENIZED_TRAIN, 'metamath')
            history_path = RESULT_ROOT / f'{tag}_training_history.csv'
            pd.DataFrame(history).assign(training_dataset=METAMATH_TRAIN_DATASET_ID,
                                         method=method, rank=rank, component=component).to_csv(history_path, index=False)
            summary.update(training_examples=len(TOKENIZED_TRAIN), init_seconds=init_seconds,
                           train_seconds=train_seconds, train_minutes=train_seconds / 60,
                           trainable_parameters=trainable, status='trained_pending_merge_eval')
            save_json(RESULT_ROOT / f'{tag}_training_time.json', summary)
            if SAVE_ADAPTERS:
                state = {name: p.detach().cpu() for name, p in model.named_parameters() if p.requires_grad}
                torch.save({'metadata': summary, 'state_dict': state}, RESULT_ROOT / f'{tag}_adapter.pt')
            torch.cuda.synchronize()
            start = time.perf_counter()
            report_progress(f'{tag}/merge', run_tag=tag)
            merged_names = merge_pissa_and_unload(model)
            torch.cuda.synchronize()
            merge_seconds = time.perf_counter() - start
            assert not any(isinstance(m, PiSSALinear) for m in model.modules()), 'Evaluation requires a merged model.'
        eval_results = evaluate_all_tasks(model, max_examples=max_eval_examples)
        if rank is not None:
            full_model_path = RESULT_ROOT / f'{tag}_full_model'
            if SAVE_FULL_MODEL:
                model = model.to('cpu')
                gc.collect()
                torch.cuda.empty_cache()
                report_progress(f'{tag}/save_model', run_tag=tag)
                model.save_pretrained(full_model_path, safe_serialization=True)
                tokenizer.save_pretrained(full_model_path)
                print('Merged full model saved:', full_model_path, flush=True)
            summary.update(merge_seconds=merge_seconds, merged_layer_count=len(merged_names),
                           evaluated_after_merge=True, full_model_saved=bool(SAVE_FULL_MODEL),
                           full_model_path=str(full_model_path) if SAVE_FULL_MODEL else None,
                           final_training_loss=history[-1]['loss'] if history else None,
                           training_history_path=str(history_path), total_parameters_with_adapter=total_params,
                           max_pissa_reconstruction_error=max(s.relative_error_stored for s in init_stats))
        summary.update(flatten_eval_results(eval_results), status='ok')
        save_summary(summary, [record for task in EVAL_TASKS for record in eval_results[task]['records']])
        return summary
    finally:
        cleanup_model(model)


In [ ]:
RUN_PLAN = [dict(spec) for spec in MANUAL_RUNS]

print('Training corpus:', METAMATH_TRAIN_DATASET_ID)
print('Evaluation tasks:', EVAL_TASKS)
print('PiSSA runs:', RUN_PLAN)

RUN_MANIFEST = {
    "cuda_allocator_config": os.environ.get("PYTORCH_ALLOC_CONF"),
    "cuda_allocator_mode": "native_defaults",
    "source_notebook_version": CURRENT_NOTEBOOK_VERSION,
    "upstream_commit": "b1521186a8a516a4e81482d5e6ad5f587d9aae52",
    "run_stage": RUN_STAGE, "seed": SEED,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "training_dataset": METAMATH_TRAIN_DATASET_ID,
    "requested_training_examples": METAMATH_TRAIN_LIMIT,
    "usable_training_examples": len(TOKENIZED_TRAIN),
    "evaluation_dataset": PISSA_EVAL_DATASET_ID,
    "evaluation_examples_per_task": EVAL_EXAMPLES_PER_TASK,
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "model_precision": "FP16", "adapter_precision": "FP32",
    "max_length": MAX_LENGTH, "max_new_tokens": EVAL_MAX_NEW_TOKENS,
    "micro_batch_size": MICRO_BATCH_SIZE, "gradient_accumulation": GRAD_ACCUM_STEPS,
    "effective_batch_size": MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "use_data_parallel": USE_DATA_PARALLEL,
    "runs": RUN_PLAN, "fresh_baseline": RUN_FRESH_BASELINE,
}
# Saved default result: original Version 11 outputs, not a newly trained checkpoint.
Q3_DEFAULT_REFERENCE = {'run_tag': 'metamath_pissa_default_r16', 'training_dataset': 'meta-math/MetaMathQA', 'training_examples': 24975, 'method': 'pissa', 'rank': 16, 'component': 'default', 'init_seconds': 162.58688042599988, 'train_seconds': 4115.782910894, 'train_minutes': 68.59638184823334, 'trainable_parameters': 11272192, 'status': 'ok', 'seed': 42, 'gsm8k_accuracy': 0.167, 'gsm8k_correct': 167, 'gsm8k_total': 1000, 'gsm8k_unparsed': 0, 'gsm8k_eval_seconds': 1245.3227264840007, 'math_accuracy': 0.041, 'math_correct': 41, 'math_total': 1000, 'math_unparsed': 550, 'math_eval_seconds': 1737.7725328470005, 'merge_seconds': 0.1636772159999964, 'merged_layer_count': 112, 'evaluated_after_merge': True, 'full_model_saved': True, 'final_training_loss': 0.4034421741962433, 'total_parameters_with_adapter': 1247086592, 'max_pissa_reconstruction_error': 0.00020509955356828868, 'source_version': 11, 'result_origin': 'reused_saved_result'}
Q3_REFERENCE_MANIFEST = {'upstream_commit': 'b1521186a8a516a4e81482d5e6ad5f587d9aae52', 'seed': 42, 'model_id': 'meta-llama/Llama-3.2-1B', 'model_revision': '4e20de362430cd3b72f300e6b0f18e50e7166e08', 'training_dataset': 'meta-math/MetaMathQA', 'requested_training_examples': 25000, 'usable_training_examples': 24975, 'evaluation_dataset': 'fxmeng/pissa-dataset', 'evaluation_examples_per_task': 1000, 'epochs': 1, 'learning_rate': 2e-05, 'model_precision': 'FP16', 'adapter_precision': 'FP32', 'max_length': 512, 'max_new_tokens': 256, 'micro_batch_size': 8, 'gradient_accumulation': 16, 'effective_batch_size': 128, 'use_data_parallel': True}
Q3_REFERENCE_PROVENANCE = {'source_version': 11, 'run_tag': 'metamath_pissa_default_r16', 'source_summary_file': 'metamath_pissa_default_r16_summary.json', 'source_summary_sha256': 'a40515007e4f15c6c201de204042921925f72831b131087ef53a6b645ec7a0a5', 'source_manifest_sha256': '34f70869a167dce1a646c58525fe667ab03c7435ec1a76e8fc4c90a92933a422'}

if RUN_STAGE == "q3_remaining":
    for key, expected_value in Q3_REFERENCE_MANIFEST.items():
        assert RUN_MANIFEST.get(key) == expected_value, f"Q3 default reference differs: {key}"
    assert Q3_DEFAULT_REFERENCE["method"] == "pissa"
    assert Q3_DEFAULT_REFERENCE["rank"] == 16 and Q3_DEFAULT_REFERENCE["component"] == "default"
    assert Q3_DEFAULT_REFERENCE["status"] == "ok" and Q3_DEFAULT_REFERENCE["seed"] == SEED
    assert Q3_DEFAULT_REFERENCE["training_examples"] == len(TOKENIZED_TRAIN)
    assert Q3_DEFAULT_REFERENCE["evaluated_after_merge"] and Q3_DEFAULT_REFERENCE["full_model_saved"]
    assert Q3_DEFAULT_REFERENCE["merged_layer_count"] == 112
    for task in EVAL_TASKS:
        assert Q3_DEFAULT_REFERENCE[f"{task}_total"] == EVAL_EXAMPLES_PER_TASK
        assert math.isclose(Q3_DEFAULT_REFERENCE[f"{task}_accuracy"],
                            Q3_DEFAULT_REFERENCE[f"{task}_correct"] / EVAL_EXAMPLES_PER_TASK,
                            rel_tol=0, abs_tol=1e-12)
    assert Q3_DEFAULT_REFERENCE["train_seconds"] > 0
    assert math.isclose(Q3_DEFAULT_REFERENCE["train_minutes"],
                        Q3_DEFAULT_REFERENCE["train_seconds"] / 60, rel_tol=1e-9)
    RUN_MANIFEST["reused_results"] = [Q3_REFERENCE_PROVENANCE]
    RUN_MANIFEST["reused_reference_manifest"] = Q3_REFERENCE_MANIFEST
    print("Q3 default result reused from Version 11; remaining components run independently.", flush=True)

save_json(RESULT_ROOT / 'run_manifest.json', RUN_MANIFEST)
PREFLIGHT_REPORT.update(status="ready", tokenizer_download=True,
    training_dataset_access=True, usable_training_examples=len(TOKENIZED_TRAIN),
    evaluation_counts={task: len(ds) for task, ds in EVAL_DATASETS.items()})
save_preflight_report()


In [ ]:
ALL_RESULTS = []
if RUN_STAGE == "q3_remaining":
    ALL_RESULTS.append(dict(Q3_DEFAULT_REFERENCE))
    save_summary(ALL_RESULTS[0])
    save_master_results(ALL_RESULTS)

def persist_run_failure(method, spec, exc):
    row = {
        "method": method,
        "run_tag": safe_run_tag(method, spec.get("rank"), spec.get("component")),
        "training_dataset": METAMATH_TRAIN_DATASET_ID, **spec,
        "status": "oom" if isinstance(exc, torch.cuda.OutOfMemoryError) else "error",
        "error_type": type(exc).__name__,
    }
    ALL_RESULTS.append(row)
    save_master_results(ALL_RESULTS)
    report_progress("run_failed", status="error", run_tag=row["run_tag"], error_type=type(exc).__name__)
    _progress_stop.set()
    print("Experiment failed:", row, flush=True)
    # Leave the traceback visible; never report a failed experiment as a successful run.
    raise exc

if RUN_FRESH_BASELINE:
    try:
        ALL_RESULTS.append(run_experiment(max_eval_examples=EVAL_EXAMPLES_PER_TASK))
        save_master_results(ALL_RESULTS)
    except Exception as exc:
        persist_run_failure("fresh", {}, exc)

for spec in RUN_PLAN:
    try:
        result = run_experiment(**spec, max_eval_examples=EVAL_EXAMPLES_PER_TASK)
        if RUN_STAGE == "q3_remaining":
            result.update(source_version=CURRENT_NOTEBOOK_VERSION, result_origin="current_execution")
            save_summary(result)
        ALL_RESULTS.append(result)
        save_master_results(ALL_RESULTS)
    except Exception as exc:
        persist_run_failure("pissa", spec, exc)

if RUN_STAGE == "q3_remaining":
    assert len(ALL_RESULTS) == 4
    assert {row["component"] for row in ALL_RESULTS} == {"default", "quarter", "half", "bottom"}
RESULTS_DF = save_master_results(ALL_RESULTS)
assert RESULTS_DF["status"].eq("ok").all()
for task in EVAL_TASKS:
    assert RESULTS_DF[f"{task}_total"].eq(EVAL_EXAMPLES_PER_TASK).all()
report_progress("all_experiments_completed", len(ALL_RESULTS), len(ALL_RESULTS), status="completed")
_progress_stop.set()
print("All selected experiments completed successfully.", flush=True)
RESULTS_DF


In [ ]:
display_cols = [
    'method', 'rank', 'component', 'gsm8k_accuracy', 'math_accuracy',
    'gsm8k_correct', 'gsm8k_total', 'math_correct', 'math_total',
    'train_minutes', 'init_seconds', 'trainable_parameters', 'gsm8k_unparsed', 'math_unparsed',
]
ok = RESULTS_DF[RESULTS_DF['status'].eq('ok')]
display(ok[[c for c in display_cols if c in ok.columns]].sort_values(
    ['method', 'rank', 'component'], na_position='first'))
pissa_only = ok[ok['method'].eq('pissa')]
if len(pissa_only):
    display(pissa_only.pivot_table(index='rank', columns='component',
                                  values=['gsm8k_accuracy', 'math_accuracy'], aggfunc='first'))


## Q3 圖表

q3_remaining 使用已驗證的 v11/default 與本次三組正式結果；四個 Rank16 components 完整且各恰好一筆才儲存 PNG、PDF、CSV。q3_source_versions.json 記錄歷史 default 與本次執行的來源版本；本次版本由保存 Kaggle notebook 的步驟指定。


In [ ]:
import matplotlib.pyplot as plt
required_components = ["default", "quarter", "half", "bottom"]
q3_rows = RESULTS_DF[
    RESULTS_DF["method"].eq("pissa") & RESULTS_DF["rank"].eq(16)
    & RESULTS_DF["status"].eq("ok")
].copy()
missing = set(required_components) - set(q3_rows["component"])
if missing:
    print("Q3 chart requires these additional components:", sorted(missing))
else:
    assert len(q3_rows) == 4 and not q3_rows["component"].duplicated().any(), "Q3 requires one row per component."
    assert set(q3_rows["component"]) == set(required_components)
    for task in EVAL_TASKS:
        assert q3_rows[f"{task}_total"].eq(EVAL_EXAMPLES_PER_TASK).all()
    q3_sources = q3_rows[["component", "run_tag"]].to_dict("records")
    for row in q3_sources:
        source = q3_rows[q3_rows["component"].eq(row["component"])].iloc[0]
        row["source_version"] = None if pd.isna(source.get("source_version")) else int(source["source_version"])
        row["result_origin"] = source.get("result_origin", "current_execution")
    save_json(RESULT_ROOT / 'q3_source_versions.json', q3_sources)
    q3_rows = q3_rows.set_index("component").loc[required_components]
    chart_data = pd.DataFrame({
        "Top": [q3_rows.loc["default", f"{t}_accuracy"] for t in EVAL_TASKS],
        "25%": [q3_rows.loc["quarter", f"{t}_accuracy"] for t in EVAL_TASKS],
        "50%": [q3_rows.loc["half", f"{t}_accuracy"] for t in EVAL_TASKS],
        "Bottom": [q3_rows.loc["bottom", f"{t}_accuracy"] for t in EVAL_TASKS],
    }, index=["GSM8K", "MATH"])
    ax = chart_data.plot.bar(figsize=(7, 4), rot=0)
    ax.set_ylabel("Accuracy")
    ax.set_title("Rank 16: singular-value components")
    ax.set_ylim(0, 1)
    ax.legend(title="Component")
    plt.tight_layout()
    plt.savefig(RESULT_ROOT / "q3_accuracy.png", dpi=200)
    plt.savefig(RESULT_ROOT / "q3_accuracy.pdf")
    chart_data.to_csv(RESULT_ROOT / "q3_accuracy.csv")
    plt.show()
